In [ ]:
import pandas as pd

df = pd.read_json('vsl_full_front.json')
df.head()

In [ ]:
import os
path_vsl_uit = 'dataset/train'

vsl_400 = df['gloss'].unique()
vsl_uit_gloss = []
for f in os.listdir(path_vsl_uit):
    if os.path.isdir(os.path.join(path_vsl_uit, f)):
        vsl_uit_gloss.append(f)

print(vsl_400)
print(vsl_uit_gloss)

In [ ]:
import re

def clean_gloss(text):
    # Xóa nội dung trong ngoặc đơn và các khoảng trắng thừa, chuyển về chữ thường
    return re.sub(r'\(.*?\)', '', text).strip().lower()

# Tạo dictionary ánh xạ từ vựng "đã làm sạch" về từ gốc
cleaned_vsl_400 = {clean_gloss(w): w for w in vsl_400}
cleaned_vsl_uit = {clean_gloss(w): w for w in vsl_uit_gloss}

print("Kết quả kiểm tra các từ lệch nhau do dấu ngoặc '()':")
found = False
for c_word, orig_400 in cleaned_vsl_400.items():
    if c_word in cleaned_vsl_uit:
        orig_uit = cleaned_vsl_uit[c_word]
        # Nếu gốc giống nhau nhưng chữ ban đầu khác nhau
        if orig_400 != orig_uit:
            print(f"- Từ gốc: '{c_word}' | vsl_400: '{orig_400}' <---> vsl_uit_gloss: '{orig_uit}'")
            found = True

if not found:
    print("Không phát hiện từ nào bị lệch chỉ do có lỗi ghi chú trong dấu ()")

# Chuyển VSL 400 thành các thư mục gloss tương ứng

In [ ]:
import json
import os
import shutil
import re

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN (TÙY CHỈNH TẠI ĐÂY)
# ==========================================
JSON_FILE_PATH = "vsl_full_front.json"      # Đường dẫn đến file json
VIDEO_SOURCE_DIR = "VSL_FULL_FRONT"         # Thư mục gốc chứa video .mp4
OUTPUT_DIR = "VSL_FULL_FRONT_CATEGORIZED"   # Thư mục đầu ra để chứa các folder phân loại

def sanitize_folder_name(name):
    """
    Làm sạch chuỗi để đảm bảo tên thư mục hợp lệ trên hệ điều hành (Windows/Linux/Mac).
    Loại bỏ các ký tự đặc biệt bị cấm như: \ / : * ? " < > |
    """
    if not isinstance(name, str):
        name = str(name)
    # Thay thế các ký tự không hợp lệ bằng khoảng trắng hoặc chuỗi rỗng
    sanitized = re.sub(r'[\\/:*?"<>|]', '', name)
    # Loại bỏ khoảng trắng thừa ở hai đầu
    return sanitized.strip()

def main():
    # 1. Tạo thư mục output nếu chưa tồn tại
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
        print(f"Đã tạo thư mục đầu ra: {OUTPUT_DIR}")

    # 2. Đọc dữ liệu từ file JSON
    try:
        with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
            vsl_data = json.load(f)
    except FileNotFoundError:
        print(f"❌ Lỗi: Không tìm thấy file JSON tại '{JSON_FILE_PATH}'. Vui lòng kiểm tra lại đường dẫn.")
        return
    except json.JSONDecodeError:
        print(f"❌ Lỗi: File '{JSON_FILE_PATH}' không đúng định dạng JSON.")
        return

    print(f"🚀 Bắt đầu xử lý {len(vsl_data)} video...\n" + "-"*40)

    success_count = 0
    missing_count = 0

    # 3. Lặp qua từng object trong JSON
    for item in vsl_data:
        video_id = item.get("video_id")
        gloss = item.get("gloss")

        if not video_id or not gloss:
            print(f"⚠️ Cảnh báo: Dữ liệu không hợp lệ (thiếu video_id hoặc gloss) -> Bỏ qua: {item}")
            continue

        # Chuẩn hóa tên thư mục theo từ khóa gloss
        folder_name = sanitize_folder_name(gloss)
        
        # Đề phòng trường hợp gloss sau khi sanitize bị rỗng
        if not folder_name:
            folder_name = "unknown_gloss"

        target_folder = os.path.join(OUTPUT_DIR, folder_name)

        # Tự động tạo folder tương ứng cho gloss nếu chưa có
        if not os.path.exists(target_folder):
            os.makedirs(target_folder)

        # Thiết lập đường dẫn file nguồn và file đích
        source_video_path = os.path.join(VIDEO_SOURCE_DIR, f"{video_id}.mp4")
        target_video_path = os.path.join(target_folder, f"{video_id}.mp4")

        # 4. Tìm và sao chép/di chuyển file video
        if os.path.exists(source_video_path):
            try:
                # Dùng shutil.copy2 để copy (giữ nguyên metadata ngày tạo/sửa của file)
                # Nếu bạn muốn DI CHUYỂN hẳn (move) thay vì copy, hãy đổi thành: shutil.move(source_video_path, target_video_path)
                shutil.move(source_video_path, target_video_path)
                print(f"✅ Đã di chuyển: '{video_id}.mp4' ---> Folder '{folder_name}'")
                success_count += 1
            except Exception as e:
                print(f"❌ Lỗi khi xử lý video {video_id}.mp4: {e}")
        else:
            print(f"⚠️ Cảnh báo: Không tìm thấy video gốc '{source_video_path}'")
            missing_count += 1

    # In báo cáo tổng kết
    print("-" * 40)
    print("🎉 KẾT THÚC TIẾN TRÌNH 🎉")
    print(f"- Số video xử lý thành công: {success_count}")
    print(f"- Số video không tìm thấy: {missing_count}")

if __name__ == "__main__":
    main()

In [ ]:
import os
import shutil

# Thư mục nguồn (chứa các folder VSL UIT)
src_uit_dir = 'dataset/train'
# Thư mục đích (nơi đã gom nhóm VSL 400)
dest_dir = 'VSL_FULL_FRONT'

# Ánh xạ thủ công tên folder từ set UIT sang tên folder tương ứng bên VSL 400
mapping_uit_to_400 = {
    "Đẹp": "Đẹp (người)"
    # Thêm các trường hợp cần ánh xạ khác vào đây nếu có (VD: "Từ A": "Từ A (ngoại lệ)")
}

if not os.path.exists(dest_dir):
    os.makedirs(dest_dir)

count_folders = 0
count_files = 0

for folder_name in os.listdir(src_uit_dir):
    src_folder_path = os.path.join(src_uit_dir, folder_name)
    
    if os.path.isdir(src_folder_path):
        # Nếu thư mục có trong mapping, đổi tên thành tên map, nếu không giữ nguyên
        dest_folder_name = mapping_uit_to_400.get(folder_name, folder_name)
        dest_folder_path = os.path.join(dest_dir, dest_folder_name)
        
        # Nếu thư mục cấu hình bên đích chưa tồn tại thì tạo mới
        if not os.path.exists(dest_folder_path):
            os.makedirs(dest_folder_path)
            
        # Lặp qua tất cả file trong thư mục của dataset/train và copy qua đích
        for file_name in os.listdir(src_folder_path):
            src_file_path = os.path.join(src_folder_path, file_name)
            dest_file_path = os.path.join(dest_folder_path, file_name)
            
            # Chỉ copy nếu là file (không copy folder con, nếu có) và file chưa tồn tại
            if os.path.isfile(src_file_path):
                if not os.path.exists(dest_file_path):
                    shutil.move(src_file_path, dest_file_path)
                    count_files += 1
                    
        count_folders += 1

print(f"Hoàn tất! Đã xử lý {count_folders} thư mục và sao chép {count_files} files sang '{dest_dir}'.")

In [ ]:
import os

dest_dir = 'VSL_FULL_FRONT'

if os.path.exists(dest_dir):
    print(f"Số lượng video trong từng gloss tại '{dest_dir}':\n" + "-"*40)
    
    gloss_counts = {}
    for folder_name in os.listdir(dest_dir):
        folder_path = os.path.join(dest_dir, folder_name)
        if os.path.isdir(folder_path):
            # Đếm số lượng file trong từng thư mục gloss
            num_videos = len([f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))])
            gloss_counts[folder_name] = num_videos
            
    # Hiển thị kết quả (sắp xếp theo tên folder cho dễ nhìn)
    for gloss, count in sorted(gloss_counts.items(), key=lambda x: x[0]):
        print(f"- {gloss}: {count} video")
        
    print("-" * 40)
    print(f"Tổng số từ (gloss): {len(gloss_counts)}")
    print(f"Tổng số video: {sum(gloss_counts.values())}")
else:
    print(f"Thư mục '{dest_dir}' chưa tồn tại.")